# LGD and EAD models
LGD uses a two-step model (logistic + linear). EAD uses a linear model on the credit conversion factor (CCF). Both are built on charged-off loans.

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, r2_score, mean_squared_error, mean_absolute_error

## Load charged-off loans
LGD and EAD only make sense for loans that defaulted. 2015–2017 loans are pooled, because there are far fewer defaults than loans.

In [10]:
df = pd.concat([pd.read_parquet("../data/train.parquet"),
                pd.read_parquet("../data/test.parquet")], ignore_index=True)

snapshot = df["last_pymnt_d"].max()
co = df[df["loan_status"] == "Charged Off"].copy()

print("Charged-off loans:", len(co))
print("Snapshot:", snapshot)

Charged-off loans: 183193
Snapshot: 2019-03-01 00:00:00


## Exposure at default and recovery rate
Balance at default = funded amount − principal repaid. CCF = balance at default ÷ funded amount. Recovery rate = (recoveries − collection fees) ÷ balance at default, kept between 0 and 1.

In [11]:
co["ead_amount"] = co["funded_amnt"] - co["total_rec_prncp"]
co["ccf"] = co["ead_amount"] / co["funded_amnt"]

co["recovery_rate"] = (co["recoveries"] - co["collection_recovery_fee"]) / co["ead_amount"]
co["recovery_rate"] = co["recovery_rate"].clip(0, 1)
co["lgd"] = 1 - co["recovery_rate"]
co["any_recovery"] = (co["recovery_rate"] > 0).astype(int)

print(co[["ccf", "recovery_rate", "lgd"]].describe().round(3))
print("\nShare with any recovery:", round(co["any_recovery"].mean(), 3))

              ccf  recovery_rate         lgd
count  183193.000     183176.000  183176.000
mean        0.714          0.088       0.912
std         0.205          0.108       0.108
min         0.000          0.000       0.000
25%         0.598          0.000       0.885
50%         0.760          0.084       0.916
75%         0.875          0.115       1.000
max         1.000          1.000       1.000

Share with any recovery: 0.648


## Recovery window
Recoveries take time to come in. For LGD, only keep loans whose last payment was at least 18 months before the snapshot (about 12+ months after charge-off). Loans that never made a payment use the issue date instead.

In [12]:
last_date = co["last_pymnt_d"].fillna(co["issue_d"])
months_since = (snapshot.year - last_date.dt.year) * 12 + (snapshot.month - last_date.dt.month)

lgd_data = co[months_since >= 18].copy()
print("LGD sample:", len(lgd_data), "of", len(co))

LGD sample: 103063 of 183193


## Features
Origination features only. Numeric columns: missing values are filled with the median, and income is logged because it's heavily skewed. Categories become dummy columns.

In [13]:
num_cols = ["loan_amnt", "term", "int_rate", "annual_inc", "dti", "fico",
            "revol_util", "emp_length", "credit_history_years", "open_acc", "total_acc"]
cat_cols = ["grade", "home_ownership", "purpose", "verification_status"]

medians = co[num_cols].median()

def make_X(df, columns=None):
    X = df[num_cols].fillna(medians)
    X["annual_inc"] = np.log1p(X["annual_inc"])  # income is very skewed
    X = pd.concat([X, pd.get_dummies(df[cat_cols].astype(str), drop_first=True, dtype=float)], axis=1)
    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0)
    return X

X_cols = make_X(co).columns

## LGD step 1: any recovery? (logistic regression)

In [14]:
X = make_X(lgd_data, X_cols)
y_any = lgd_data["any_recovery"]
y_rate = lgd_data["recovery_rate"]

X_tr, X_te, ya_tr, ya_te, yr_tr, yr_te = train_test_split(
    X, y_any, y_rate, test_size=0.2, random_state=42
)

lgd_step1 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
lgd_step1.fit(X_tr, ya_tr)

p_any = lgd_step1.predict_proba(X_te)[:, 1]
print("Step 1 Gini:", round(2 * roc_auc_score(ya_te, p_any) - 1, 3))

Step 1 Gini: 0.198


## LGD step 2: how much is recovered? (linear regression)
Trained only on loans with some recovery.

In [15]:
lgd_step2 = make_pipeline(StandardScaler(), LinearRegression())
lgd_step2.fit(X_tr[ya_tr == 1], yr_tr[ya_tr == 1])

rate_pred = lgd_step2.predict(X_te[ya_te == 1]).clip(0, 1)
rate_true = yr_te[ya_te == 1]

print("Step 2 R2:  ", round(r2_score(rate_true, rate_pred), 3))
print("Step 2 RMSE:", round(np.sqrt(mean_squared_error(rate_true, rate_pred)), 3))
print("Step 2 MAE: ", round(mean_absolute_error(rate_true, rate_pred), 3))

Step 2 R2:   0.011
Step 2 RMSE: 0.111
Step 2 MAE:  0.07


## Combined LGD
Expected recovery = P(any recovery) × predicted recovery rate. LGD = 1 − expected recovery.

In [16]:
def predict_lgd(X):
    p = lgd_step1.predict_proba(X)[:, 1]
    rate = lgd_step2.predict(X).clip(0, 1)
    return (1 - p * rate).clip(0, 1)

lgd_pred = predict_lgd(X_te)
lgd_true = 1 - yr_te

print("Mean predicted LGD:", round(lgd_pred.mean(), 3))
print("Mean actual LGD:   ", round(lgd_true.mean(), 3))
print("MAE:               ", round(mean_absolute_error(lgd_true, lgd_pred), 3))

by_grade = pd.DataFrame({"grade": lgd_data.loc[X_te.index, "grade"],
                         "actual": lgd_true, "predicted": lgd_pred})
print(by_grade.groupby("grade")[["actual", "predicted"]].mean().round(3))

Mean predicted LGD: 0.9
Mean actual LGD:    0.899
MAE:                0.071
       actual  predicted
grade                   
A       0.902      0.898
B       0.898      0.900
C       0.899      0.900
D       0.901      0.901
E       0.898      0.902
F       0.897      0.901
G       0.905      0.902


## EAD model (linear regression on CCF)
Uses all charged-off loans, since the balance at default is known as soon as a loan charges off.

In [18]:
X_ead = make_X(co, X_cols)
y_ccf = co["ccf"]

Xe_tr, Xe_te, ye_tr, ye_te = train_test_split(X_ead, y_ccf, test_size=0.2, random_state=42)

ead_model = make_pipeline(StandardScaler(), LinearRegression())
ead_model.fit(Xe_tr, ye_tr)

ccf_pred = ead_model.predict(Xe_te).clip(0, 1)

print("EAD R2:  ", round(r2_score(ye_te, ccf_pred), 3))
print("EAD RMSE:", round(np.sqrt(mean_squared_error(ye_te, ccf_pred)), 3))
print("EAD MAE: ", round(mean_absolute_error(ye_te, ccf_pred), 3))
print("Mean predicted CCF:", round(ccf_pred.mean(), 3), "| actual:", round(ye_te.mean(), 3))

EAD R2:   0.21
EAD RMSE: 0.182
EAD MAE:  0.143
Mean predicted CCF: 0.713 | actual: 0.713


## EAD benchmark: payment schedule
For amortizing loans, the balance at default can also be worked out from the payment schedule, using how many months were paid before default. If the regression doesn't beat this, the schedule is the better EAD method.

In [19]:
test_loans = co.loc[Xe_te.index]
r = test_loans["int_rate"] / 1200
k = ((test_loans["last_pymnt_d"].dt.year - test_loans["issue_d"].dt.year) * 12 +
     (test_loans["last_pymnt_d"].dt.month - test_loans["issue_d"].dt.month)).fillna(0)

P, pmt = test_loans["funded_amnt"], test_loans["installment"]
balance = P * (1 + r) ** k - pmt * ((1 + r) ** k - 1) / r
ccf_schedule = (balance / P).clip(0, 1)

print("Schedule MAE:  ", round(mean_absolute_error(ye_te, ccf_schedule), 3))
print("Regression MAE:", round(mean_absolute_error(ye_te, ccf_pred), 3))

Schedule MAE:   0.019
Regression MAE: 0.143


## Save
Save the models and metrics for the ECL notebook and the Streamlit app.

In [20]:
metrics = pd.DataFrame({
    "metric": ["LGD step 1 Gini", "LGD step 2 R2", "LGD MAE", "Mean pred LGD", "Mean actual LGD",
               "EAD R2", "EAD MAE", "Schedule EAD MAE"],
    "value": [2 * roc_auc_score(ya_te, p_any) - 1, r2_score(rate_true, rate_pred),
              mean_absolute_error(lgd_true, lgd_pred), lgd_pred.mean(), lgd_true.mean(),
              r2_score(ye_te, ccf_pred), mean_absolute_error(ye_te, ccf_pred),
              mean_absolute_error(ye_te, ccf_schedule)],
}).round(3)
print(metrics)

metrics.to_csv("../artifacts/lgd_ead_metrics.csv", index=False)
joblib.dump({"lgd_step1": lgd_step1, "lgd_step2": lgd_step2, "ead_model": ead_model,
             "X_cols": list(X_cols), "medians": medians,
             "num_cols": num_cols, "cat_cols": cat_cols},
            "../artifacts/lgd_ead_models.pkl")

             metric  value
0   LGD step 1 Gini  0.198
1     LGD step 2 R2  0.011
2           LGD MAE  0.071
3     Mean pred LGD  0.900
4   Mean actual LGD  0.899
5            EAD R2  0.210
6           EAD MAE  0.143
7  Schedule EAD MAE  0.019


['../artifacts/lgd_ead_models.pkl']

## Findings
- **LGD is well calibrated but weakly discriminating.** Mean predicted LGD is 0.900 vs 0.899 actual, but step 1 Gini (0.20) and step 2 R² (0.01) are low, so the model is close to a flat ~90% LGD.
- **Why:** LC sold charged-off loans to debt buyers, so recoveries are low and similar across borrowers. Origination features can't explain much of that.
- **EAD: the payment schedule beats the regression** (MAE 0.019 vs 0.143), so the ECL uses the schedule for EAD.
- **Caveat:** the schedule benchmark uses the actual default month, which is only known in hindsight. The lifetime ECL steps through each future month, though, so this is how the schedule would be used anyway.

**Implications**
- With LGD flat at ~90%, credit losses on this book are driven almost entirely by PD. Underwriting and pricing decisions should focus on default risk, not on recovery assumptions.
- A 90% LGD also means collateral-free consumer lending leaves almost no loss cushion after default. Every 1pp of default rate costs ~0.9pp of balance, which the interest rate has to cover.
- Using the payment schedule for EAD removes one source of model error from the ECL at no extra complexity.